# Multimodal Extraction — Images to Typed Python Objects

**Week 2 | Notebook 4 of 4**

**What you'll learn:**
- Outlines vision model setup
- Extracting tables from scanned invoices
- Structured output from charts and graphs
- Comparing extraction accuracy: Outlines vs. raw GPT-4V prompting
- Batch document processing pipeline

**Runtime:** ~40 minutes

**Note:** This notebook uses vision models. Ensure you have access to GPT-4o or GPT-4V.

In [1]:
# 💰 COST ESTIMATE
from src.cost_tracker import print_cost_warning

print_cost_warning("02_outlines/04_vision_structure.ipynb")

💰 COST ESTIMATE
----------------------------------------
Notebook:  02_outlines/04_vision_structure.ipynb
Task:      Vision + structured output
Calls:     ~10

With GPT-4o:       $0.25 USD
With GPT-4o-mini:  $0.03 USD (10x cheaper)
With Ollama:       $0.00 USD (free, local)

💡 TIP: Set USE_SMALL_MODEL=true or USE_OLLAMA=true in .env to save money.
----------------------------------------


## 1. Setup

In [2]:
import base64
import json
from glob import glob
from pathlib import Path
from typing import Literal

from outlines.inputs import Chat, Image
from PIL import Image as PILImage
from pydantic import BaseModel

from src.config import get_outlines_model

# Vision requires a vision-capable model: gpt-4o (openai) and gemini-3.7-flash (gemini) work.
# Groq's gpt-oss-120b has no vision — switch LLM_PROVIDER or keep openai for this notebook.
model = get_outlines_model()

## 2. Extracting Tables from Scanned Invoices

**Real document samples.** The images used below are *real* documents downloaded from
[Wikimedia Commons](https://commons.wikimedia.org/) and checked into `data/invoice_samples/`:

| File | Document | Author | License |
|---|---|---|---|
| `bill_green_tangerine.jpg` | Restaurant bill, Green Tangerine, Hanoi | Alpha (Flickr) | [CC BY-SA 2.0](https://creativecommons.org/licenses/by-sa/2.0/) |
| `invoice_kanazawa_restaurant.jpg` | Handwritten restaurant bill, Kanazawa, Japan | Tbatb | [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/) |
| `invoice_dairymans_league.jpg` | Purchase-order confirmation, Dairymen's League, 1925 | Brian Fowler | Public domain |
| `chart_energy_safety.png` | "Safest and cleanest sources of energy", Our World in Data | Max Roser / Our World in Data | [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/) |
| `chart_life_expectancy_europe.png` | Healthy life expectancy, Europe (WHO data) | Lady3mlnm | [CC0](https://creativecommons.org/publicdomain/zero/1.0/) |

If the files are missing, the notebook falls back to small synthetic stand-ins — but real
scans (handwriting, stamps, creases, mixed languages) are what make the extraction interesting.

In [7]:
# Real scanned bill: a handwritten restaurant bill from Hanoi (see attribution above).
# In production, point this at your own scans: INVOICE_IMG = Path("invoice_scan.png")
SAMPLES = Path("../../data/invoice_samples")
INVOICE_IMG = SAMPLES / "sample.png"


class LineItem(BaseModel):
    description: str
    quantity: float
    price: float


class Invoice(BaseModel):
    vendor: str
    invoice_number: str
    date: str
    currency: str
    line_items: list[LineItem]
    total: float


if INVOICE_IMG.exists():
    img = PILImage.open(INVOICE_IMG)
else:
    # Fallback: downloaded assets missing — regenerate the synthetic sample
    from PIL import ImageDraw

    img = PILImage.new("RGB", (400, 200), color="white")
    draw = ImageDraw.Draw(img)
    draw.text((10, 10), "Invoice #1234", fill="black")
    draw.text((10, 40), "Item    Qty    Price", fill="black")
    draw.text((10, 70), "WidgetA   2     $50", fill="black")
    draw.text((10, 100), "WidgetB   1     $30", fill="black")
    img.save("./sample_invoice.png")
    # outlines.inputs.Image requires img.format to be set — reload from the saved file
    img = PILImage.open("./sample_invoice.png")

prompt = Chat(
    [
        {"role": "system", "content": "Extract structured data from document images."},
        {"role": "user", "content": ["Extract the itemized table from this bill:", Image(img)]},
    ]
)

result = model(prompt, Invoice)
invoice = Invoice.model_validate_json(result)

print(f"Vendor:   {invoice.vendor}")
print(f"Bill no.: {invoice.invoice_number}   Date: {invoice.date}   Currency: {invoice.currency}")
print(f"{'Item':<30} {'Qty':>4} {'Price':>10}")
for item in invoice.line_items:
    print(f"{item.description:<30} {item.quantity:>4.0f} {item.price:>10.0f}")
print(f"{'TOTAL':<35} {invoice.total:>10.0f}")

Vendor:   Thynk Unlimited
Bill no.: 01234   Date: 11.02.2030   Currency: USD
Item                            Qty      Price
Brand consultation                1        100
logo design                       1        100
Website design                    1        100
Social media templates            1        100
Brand photography                 1        100
Brand guide                       1        100
TOTAL                                      440


## 3. Structured Output from Charts and Graphs

In [4]:
# Real published chart: "What are the safest and cleanest sources of energy?"
# by Our World in Data (CC BY 4.0) — see attribution table in Section 2.
CHART_IMG = SAMPLES / "chart_energy_safety.png"


class DataPoint(BaseModel):
    label: str
    value: float


class ChartData(BaseModel):
    chart_type: Literal["bar", "line", "pie", "scatter", "other"]  # closed set — guaranteed enum
    title: str
    x_axis_label: str
    y_axis_label: str
    unit: str
    source: str
    data_points: list[DataPoint]  # typed — bare `dict` is rejected by OpenAI structured outputs


if CHART_IMG.exists():
    chart_img = PILImage.open(CHART_IMG)
else:
    # Fallback: downloaded assets missing — regenerate the synthetic bar chart
    from PIL import ImageDraw

    chart_img = PILImage.new("RGB", (400, 300), color="white")
    draw = ImageDraw.Draw(chart_img)
    draw.text((10, 10), "Sales by Quarter", fill="black")
    draw.text((50, 250), "Q1  Q2  Q3  Q4", fill="black")
    draw.rectangle([(50, 150), (100, 250)], fill="blue")  # Q1: 100
    draw.rectangle([(120, 100), (170, 250)], fill="blue")  # Q2: 150
    draw.rectangle([(190, 80), (240, 250)], fill="blue")  # Q3: 170
    draw.rectangle([(260, 120), (310, 250)], fill="blue")  # Q4: 130
    chart_img.save("./sample_chart.png")
    # outlines.inputs.Image requires img.format to be set — reload from the saved file
    chart_img = PILImage.open("./sample_chart.png")

prompt = Chat(
    [
        {
            "role": "user",
            "content": ["Extract the chart's data series as structured output:", Image(chart_img)],
        }
    ]
)

result = model(prompt, ChartData)
chart = ChartData.model_validate_json(result)

print(f"Chart type: {chart.chart_type}")
print(f"Title: {chart.title}")
print(f"Unit: {chart.unit}   Source: {chart.source}")
print(f"Data points: {len(chart.data_points)}")
for point in chart.data_points:
    print(f"  {point.label}: {point.value}")

Chart type: bar
Title: Energy Sources Safety and Cleanliness
Unit: deaths per TWh and tonnes CO₂-eq per GWh   Source: Our World in Data
Data points: 16
  Coal (Death Rate): 24.6
  Oil (Death Rate): 18.4
  Natural Gas (Death Rate): 2.8
  Biomass (Death Rate): 4.6
  Hydropower (Death Rate): 1.3
  Wind (Death Rate): 0.04
  Nuclear energy (Death Rate): 0.03
  Solar (Death Rate): 0.02
  Coal (Emissions): 820.0
  Oil (Emissions): 720.0
  Natural Gas (Emissions): 490.0
  Biomass (Emissions): 154.0
  Hydropower (Emissions): 34.0
  Wind (Emissions): 4.0
  Nuclear energy (Emissions): 3.0
  Solar (Emissions): 5.0


## 4. Comparing Extraction Accuracy: Outlines vs Raw GPT-4V

The raw call below goes through `src.config` (`get_openai_client()` / `get_model()`) — the
same provider and model the Outlines calls used — so the comparison is apples-to-apples, and the
cost toggles in `.env` keep working.

In [5]:
# Raw prompting approach (no structure guarantee)
from src.config import get_model, get_openai_client

client = get_openai_client()
model_name = get_model()

with open(INVOICE_IMG, "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode("utf-8")

raw_response = client.chat.completions.create(
    model=model_name,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Extract the table data as JSON with headers and rows"},
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
            ],
        }
    ],
)

raw_text = raw_response.choices[0].message.content
print("--- Raw model output ---")
print(raw_text)
print("------------------------")

try:
    json.loads(raw_text)
    print("This time it happened to be parseable.")
except json.JSONDecodeError as e:
    print(f"json.loads failed: {e}")
    print("Raw prompting usually needs cleanup (markdown fences, prose) before parsing.")

print()
print("Raw approach: May produce invalid JSON or markdown formatting")
print("Outlines approach: Guaranteed valid JSON matching schema")
print("\nFor vision tasks, Outlines works best with text-heavy images.")
print("For complex visual reasoning, raw prompting + manual validation may be needed.")

--- Raw model output ---
```json
{
  "headers": ["Items", "Quantity", "Prices"],
  "rows": [
    {
      "Items": "Fish raviolis",
      "Quantity": 1,
      "Prices": 169
    },
    {
      "Items": "Mango crisp",
      "Quantity": 1,
      "Prices": 169
    },
    {
      "Items": "Slice duck",
      "Quantity": 1,
      "Prices": ""
    },
    {
      "Items": "Bullet",
      "Quantity": 1,
      "Prices": ""
    },
    {
      "Items": "Lait(s)",
      "Quantity": 1,
      "Prices": 30
    }
  ]
}
```
------------------------
json.loads failed: Expecting value: line 1 column 1 (char 0)
Raw prompting usually needs cleanup (markdown fences, prose) before parsing.

Raw approach: May produce invalid JSON or markdown formatting
Outlines approach: Guaranteed valid JSON matching schema

For vision tasks, Outlines works best with text-heavy images.
For complex visual reasoning, raw prompting + manual validation may be needed.


## 5. Batch Document Processing Pipeline

In [8]:
class DocumentExtraction(BaseModel):
    document_type: str
    vendor: str
    invoice_number: str
    date: str
    currency: str
    total_amount: float | None  # some documents (e.g. purchase orders) carry no total


# Process every real document sample in the directory
image_paths = sorted(glob(str(SAMPLES / "invoice_*.jpg")) + glob(str(SAMPLES / "bill_*.jpg")))

results = []
for path in image_paths:
    img = PILImage.open(path)
    prompt = Chat([{"role": "user", "content": ["Extract invoice data:", Image(img)]}])
    result = model(prompt, DocumentExtraction)
    doc = DocumentExtraction.model_validate_json(result)
    results.append(doc)
    print(f"Processed {Path(path).name}: {doc.document_type} — {doc.vendor} ({doc.invoice_number})")

print(f"\nTotal documents processed: {len(results)}")
for doc in results:
    total = f"{doc.currency} {doc.total_amount}" if doc.total_amount is not None else "no total"
    print(f"  {doc.document_type:<28} {total}")

Processed bill_green_tangerine.jpg: Invoice — Green Tangerine Bar & Restaurant (002911)
Processed invoice_dairymans_league.jpg: Purchase Order — Dairymen's League Co-operative Association, Inc. (WHE-8381-C)
Processed invoice_kanazawa_restaurant.jpg: Invoice — Unknown (Unknown)

Total documents processed: 3
  Invoice                      VND 368000.0
  Purchase Order               no total
  Invoice                      JPY 2970.0
